# 💬 Lesson 5.2 — Large Language Models and RAG

**AI Crash Course for Absolute Beginners**

In this notebook:
- Generate text with a local GPT-2 model
- Build word and sentence embeddings from scratch intuition
- Implement Retrieval-Augmented Generation (RAG) from scratch
- Use OpenAI API (optional) for production-quality responses

> **Note:** OpenAI API cells require your own API key. The RAG section works entirely free with local models.

> Run each cell with **Shift + Enter**.

In [ ]:
!pip install transformers torch numpy matplotlib scikit-learn sentence-transformers --quiet
# sentence-transformers — a library specifically for creating sentence-level embeddings
# It wraps transformer models and makes them easy to use for semantic similarity tasks

import numpy as np
import matplotlib.pyplot as plt
import torch
print("Ready!")

---
## Part 1 — Text Generation with GPT-2

In [ ]:
from transformers import pipeline

# pipeline("text-generation") loads a language model that predicts the next words
# model="gpt2" is a small but capable model (117M parameters) that runs locally — no API key needed
# device=-1 forces it to run on CPU (use device=0 for GPU)
generator = pipeline("text-generation", model="gpt2", device=-1)

prompts = [
    "Artificial intelligence will transform healthcare by",
    "The most important thing a beginner should know about machine learning is"
]

for prompt in prompts:
    # max_new_tokens — how many new words to generate (does not count the prompt)
    # do_sample=True — randomly sample from probability distribution (more creative)
    # temperature=0.8 — controls randomness: lower = more predictable, higher = more creative
    # pad_token_id=50256 — tells the model what token to use for padding (avoids a warning)
    result = generator(prompt, max_new_tokens=60, num_return_sequences=1,
                       do_sample=True, temperature=0.8, pad_token_id=50256)
    print(f"PROMPT: {prompt}")
    print(f"OUTPUT: {result[0]['generated_text']}")
    print("-" * 60)

In [ ]:
# Effect of temperature on creativity vs consistency
prompt = "A robot entered the room and"
print(f"Prompt: '{prompt}'\n")

for temp in [0.1, 0.7, 1.5]:
    result = generator(prompt, max_new_tokens=40, do_sample=True,
                       temperature=temp, pad_token_id=50256,
                       num_return_sequences=1)
    text = result[0]["generated_text"][len(prompt):].strip()
    print(f"Temperature={temp}: ...{text}")

---
## Part 2 — Word Embeddings: Words as Vectors

In [ ]:
# Simple 2D embeddings for intuition (real ones are 1536 dimensions)
# Each word is represented as a point in 2D space — similar words are close together
word_vectors = {
    "king"   : np.array([0.9, 0.8]),
    "queen"  : np.array([0.9, 0.3]),
    "man"    : np.array([0.5, 0.8]),
    "woman"  : np.array([0.5, 0.3]),
    "doctor" : np.array([0.7, 0.6]),
    "hospital": np.array([0.3, 0.6]),
    "puppy"  : np.array([0.1, 0.2]),
    "kitten" : np.array([0.2, 0.1]),
}

def cosine_sim(a, b):
    # Cosine similarity measures the angle between two vectors (ignores length)
    # Result is between -1 (opposite) and 1 (identical direction), 0 = unrelated
    # np.dot = dot product | np.linalg.norm = vector length (magnitude)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

pairs = [("king","queen"), ("king","puppy"), ("doctor","hospital"), ("man","woman")]
print("Cosine similarities:")
for w1, w2 in pairs:
    sim = cosine_sim(word_vectors[w1], word_vectors[w2])
    print(f"  {w1:10} - {w2:10}: {sim:.3f}")

# Plot
fig, ax = plt.subplots(figsize=(6, 5))
for word, vec in word_vectors.items():
    ax.scatter(*vec, s=100)   # *vec unpacks [x, y] into two separate arguments
    ax.annotate(word, vec, textcoords="offset points", xytext=(6, 3), fontsize=10)
ax.set_xlabel("Dimension 1 (royalty?)")
ax.set_ylabel("Dimension 2 (maleness?)")
ax.set_title("2D Word Vectors — similar words cluster together")
plt.tight_layout()
plt.show()

---
## Part 3 — Sentence Embeddings with sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

# SentenceTransformer wraps a transformer model to produce one embedding vector per sentence
# "all-MiniLM-L6-v2" is small (80MB), fast, and great for semantic similarity
model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "How do I train a neural network?",
    "What steps are needed to build a deep learning model?",
    "What is the weather like today?",
    "Tell me about gradient descent.",
    "My cat knocked over my coffee."
]

# model.encode() converts each sentence into a fixed-size vector (384 dimensions here)
# Sentences with similar meaning will have vectors pointing in similar directions
embeddings = model.encode(sentences)
print(f"Embedding shape: {embeddings.shape}  (5 sentences x {embeddings.shape[1]} dims)")

# cosine_similarity computes the similarity between every pair of sentences
# Result is a 5x5 matrix — row i, column j = similarity between sentence i and sentence j
from sklearn.metrics.pairwise import cosine_similarity
sim_matrix = cosine_similarity(embeddings)

plt.figure(figsize=(8, 6))
plt.imshow(sim_matrix, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="Cosine similarity")
short_labels = [s[:30] + "..." if len(s) > 30 else s for s in sentences]
plt.xticks(range(5), short_labels, rotation=45, ha="right", fontsize=8)
plt.yticks(range(5), short_labels, fontsize=8)
plt.title("Sentence Embedding Similarities")
plt.tight_layout()
plt.show()

---
## Part 4 — RAG: Retrieval-Augmented Generation from Scratch

In [ ]:
# Step 1: Build a knowledge base (the 'database')
knowledge_base = [
    "Backpropagation is an algorithm for training neural networks by computing gradients of the loss function with respect to each weight using the chain rule.",
    "The transformer architecture uses self-attention to relate different positions of a sequence to each other, enabling parallel processing unlike RNNs.",
    "Overfitting occurs when a model learns the training data too well, capturing noise rather than the underlying pattern, leading to poor generalisation.",
    "Convolutional neural networks use shared filter weights to detect local patterns in images, making them parameter-efficient and translation-invariant.",
    "Transfer learning involves taking a model trained on a large dataset and adapting it to a new, smaller dataset, saving computation and improving results.",
    "RLHF (Reinforcement Learning from Human Feedback) is used to align LLMs with human preferences by training a reward model on human rankings.",
    "The attention mechanism computes a weighted sum of values, where weights are determined by the compatibility between a query and each key.",
    "Gradient descent is an optimisation algorithm that iteratively moves in the direction of steepest descent of the loss function to find a minimum.",
    "Batch normalisation normalises the inputs to each layer, accelerating training and allowing higher learning rates.",
    "Dropout randomly sets a fraction of activations to zero during training, acting as an ensemble method to prevent overfitting."
]

# Step 2: Encode all chunks into vectors
chunk_embeddings = model.encode(knowledge_base)
print(f"Knowledge base: {len(knowledge_base)} chunks")
print(f"Embedding shape: {chunk_embeddings.shape}")

In [ ]:
# Step 3: RAG retrieval function
def retrieve(query, top_k=2):
    """Find the most relevant chunks for a query using semantic similarity."""
    query_emb = model.encode([query])   # encode query as a vector (in a list so shape is [1, 384])
    # cosine_similarity computes similarity between the query and every knowledge base chunk
    # Result shape: (1, n_chunks) — we take [0] to get a 1D array of similarity scores
    similarities = cosine_similarity(query_emb, chunk_embeddings)[0]
    # .argsort() returns indices that would sort the array from lowest to highest
    # [::-1] reverses it (highest first), [:top_k] takes just the top K
    top_indices = similarities.argsort()[::-1][:top_k]
    return [(knowledge_base[i], similarities[i]) for i in top_indices]

# Step 4: Generate answer with context
def rag_answer(question):
    """Simple RAG: retrieve relevant context, then generate."""
    retrieved = retrieve(question, top_k=2)

    # Build the context string by joining the retrieved chunks
    context = "\n".join([f"- {chunk}" for chunk, score in retrieved])
    # The prompt includes the retrieved context so the model can ground its answer in facts
    prompt  = f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"

    # Generate with GPT-2 (in production, use a better model like GPT-4 for coherent answers)
    # do_sample=False means greedy decoding — always pick the most likely next word (consistent)
    result = generator(prompt, max_new_tokens=80, pad_token_id=50256,
                       do_sample=False, num_return_sequences=1)
    # Slice off the original prompt from the generated text to get just the answer
    answer = result[0]["generated_text"][len(prompt):].strip()

    return context, answer


questions = [
    "What is backpropagation?",
    "How does attention work in transformers?",
    "Why do models overfit?"
]

for q in questions:
    ctx, ans = rag_answer(q)
    print(f"Question: {q}")
    print(f"Retrieved context:\n{ctx}")
    print(f"Generated answer: {ans}")
    print("-" * 60)

---
## Part 5 — Using OpenAI API (optional, requires API key)

In [ ]:
# Uncomment and fill in your API key to run this cell

# import openai
# client = openai.OpenAI(api_key="sk-...")  # your key here
#
# def rag_with_gpt(question):
#     retrieved = retrieve(question, top_k=3)
#     context = "\n".join([f"- {chunk}" for chunk, _ in retrieved])
#
#     response = client.chat.completions.create(
#         model="gpt-4o-mini",
#         messages=[
#             {"role": "system", "content": "You are a helpful AI tutor. Answer using only the provided context."},
#             {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {question}"}
#         ]
#     )
#     return response.choices[0].message.content
#
# print(rag_with_gpt("How does backpropagation work?"))

print("OpenAI API cell is commented out to protect your key.")
print("Fill in your API key above and uncomment to run.")

---
## RAG Pipeline Summary

```
User Question
    |
    v
Embed question  (sentence-transformers)
    |
    v
Vector search   (cosine similarity against knowledge base)
    |
    v
Top-K chunks retrieved
    |
    v
Prompt = system + context + question
    |
    v
LLM generates answer  (grounded in retrieved facts)
```

---
## Challenge Exercises

1. **Expand the knowledge base**: Add 10 more facts about AI topics you know. Run the same questions and see if retrieval improves.
2. **Chunking strategy**: Split a Wikipedia article into sentences and use those as your knowledge base.
3. **Reranking**: After retrieving top-10 chunks, use a cross-encoder to rerank them before passing to the LLM.

---
**Next lesson:** [Lesson 5.3 — Prompt Engineering](https://colab.research.google.com/github/GirlEf/ai-crash-course/blob/main/notebooks/lesson-5.3-prompting.ipynb)